In [1]:
from google.colab import files
uploaded = files.upload()

Saving Cleaned_Car_data.csv to Cleaned_Car_data.csv


In [2]:
import pandas as pd
import numpy as np

In [3]:
car = pd.read_csv('Cleaned_Car_data.csv')
print(car.head())

   Unnamed: 0                    name   company  year   Price  kms_driven  \
0           0     Hyundai Santro Xing   Hyundai  2007   80000       45000   
1           1     Mahindra Jeep CL550  Mahindra  2006  425000          40   
2           2       Hyundai Grand i10   Hyundai  2014  325000       28000   
3           3  Ford EcoSport Titanium      Ford  2014  575000       36000   
4           4               Ford Figo      Ford  2012  175000       41000   

  fuel_type  
0    Petrol  
1    Diesel  
2    Petrol  
3    Diesel  
4    Diesel  


In [4]:
car.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 816 entries, 0 to 815
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   Unnamed: 0  816 non-null    int64 
 1   name        816 non-null    object
 2   company     816 non-null    object
 3   year        816 non-null    int64 
 4   Price       816 non-null    int64 
 5   kms_driven  816 non-null    int64 
 6   fuel_type   816 non-null    object
dtypes: int64(4), object(3)
memory usage: 44.8+ KB


In [5]:
car.shape

(816, 7)

In [6]:
car.head()

,Unnamed: 0,name,company,year,Price,kms_driven,fuel_type
0,0,Hyundai Santro Xing,Hyundai,2007,80000,45000,Petrol
1,1,Mahindra Jeep CL550,Mahindra,2006,425000,40,Diesel
2,2,Hyundai Grand i10,Hyundai,2014,325000,28000,Petrol
3,3,Ford EcoSport Titanium,Ford,2014,575000,36000,Diesel
4,4,Ford Figo,Ford,2012,175000,41000,Diesel


In [7]:
car=car[car['Price']<6e6]

#Model

In [8]:
X=car.drop(columns='Price')
Y=car['Price']

In [9]:
from sklearn.model_selection import train_test_split
X_train,X_test,Y_train,Y_test=train_test_split(X,Y,test_size=0.2)

In [10]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import make_column_transformer
from sklearn.pipeline import make_pipeline

In [11]:
ohe=OneHotEncoder()
ohe.fit(X[['name','company','fuel_type']])

OneHotEncoder()

In [12]:
from math import remainder
column_trans=make_column_transformer((OneHotEncoder(categories=ohe.categories_),['name','company','fuel_type']),remainder='passthrough')

In [13]:
lr=LinearRegression()

In [14]:
pipe=make_pipeline(column_trans,lr)

In [15]:
pipe.fit(X_train,Y_train)

/usr/local/lib/python3.12/dist-packages/sklearn/compose/_column_transformer.py:1667: FutureWarning: 
The format of the columns of the 'remainder' transformer in ColumnTransformer.transformers_ will change in version 1.7 to match the format of the other transformers.
At the moment the remainder columns are stored as indices (of type int). With the same ColumnTransformer configuration, in the future they will be stored as column names (of type str).
To use the new behavior now and suppress this warning, use ColumnTransformer(force_int_remainder_cols=False).

  warnings.warn(


Pipeline(steps=[('columntransformer',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('onehotencoder',
                                                  OneHotEncoder(categories=[array(['Audi A3 Cabriolet', 'Audi A4 1.8', 'Audi A4 2.0', 'Audi A6 2.0',
       'Audi A8', 'Audi Q3 2.0', 'Audi Q5 2.0', 'Audi Q7', 'BMW 3 Series',
       'BMW 5 Series', 'BMW 7 Series', 'BMW X1', 'BMW X1 sDrive20d',
       'BMW X1 xDrive20d', 'Chevrolet Beat', 'Chevrolet Beat...
                                                                            array(['Audi', 'BMW', 'Chevrolet', 'Datsun', 'Fiat', 'Force', 'Ford',
       'Hindustan', 'Honda', 'Hyundai', 'Jaguar', 'Jeep', 'Land',
       'Mahindra', 'Maruti', 'Mercedes', 'Mini', 'Mitsubishi', 'Nissan',
       'Renault', 'Skoda', 'Tata', 'Toyota', 'Volkswagen', 'Volvo'],
      dtype=object),
                                                                            array(['Diesel', 'LPG', 'Petrol'], dtype=object)]),
                                                  ['name', 'company',
                                                   'fuel_type'])])),
                ('linearregression', LinearRegression())])

In [16]:
y_pred=pipe.predict(X_test)

In [17]:
r2_score(Y_test,y_pred)

0.5768036314766094

In [18]:
scores=[]
for i in range(1000):
  X_train,X_test,Y_train,Y_test=train_test_split(X,Y,test_size=0.2,random_state=i)
  lr=LinearRegression()
  pipe=make_pipeline(column_trans,lr)
  pipe.fit(X_train,Y_train)
  y_pred=pipe.predict(X_test)
  scores.append(r2_score(Y_test,y_pred))

In [19]:
np.argmax(scores)

np.int64(433)

In [20]:
scores[np.argmax(scores)]

0.845601022823402

In [21]:
X_train,X_test,Y_train,Y_test=train_test_split(X,Y,test_size=0.2,random_state=np.argmax(scores))
lr=LinearRegression()
pipe=make_pipeline(column_trans,lr)
pipe.fit(X_train,Y_train)
y_pred=pipe.predict(X_test)
r2_score(Y_test,y_pred)

0.845601022823402

In [22]:
import pickle

In [24]:
pickle.dump(pipe,open('LinerRegressionModel.pkl','wb'))

In [33]:
pipe.predict(pd.DataFrame([[0,'Hyundai Grand i10','Hyundai',2014,28000,'Petrol']],columns=['Unnamed: 0','name','company','year','kms_driven','fuel_type']))

array([212200.02697813])